# Code 10.6.3: Measuring the variance of the gradient estimate

For the current, frozen policy, samples `n` episodes, computes the flattened gradient of the surrogate loss for each episode with both weightings, and returns the total variance (the trace of the covariance matrix) of each set of estimates. It reuses the definitions of Code 10.6.2.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.
# From Code 10.6.2
import numpy as np

import torch

def rollout(env, policy, seed=None):
    """Play one episode with the current policy; return states, actions, rewards."""
    s, _ = env.reset(seed=seed)
    states, actions, rewards = [], [], []
    done = False
    while not done:
        with torch.no_grad():
            a = int(policy.dist(torch.as_tensor(s, dtype=torch.float32)).sample())
        s2, r, terminated, truncated, _ = env.step(a)
        states.append(s)
        actions.append(a)
        rewards.append(r)
        s, done = s2, terminated or truncated
    return (torch.as_tensor(np.array(states), dtype=torch.float32),
            torch.as_tensor(actions), np.array(rewards, dtype=np.float32))

def rewards_to_go(rewards, gamma):
    """G_t = r_{t+1} + gamma r_{t+2} + ... for every step of one episode."""
    G, out = 0.0, np.zeros_like(rewards)
    for t in reversed(range(len(rewards))):
        G = rewards[t] + gamma * G
        out[t] = G
    return torch.as_tensor(out)

def policy_loss(policy, states, actions, weights):
    """Surrogate whose gradient is the REINFORCE estimate: -sum_t log pi(a_t|s_t) w_t."""
    logp = policy.dist(states).log_prob(actions)
    return -(logp * weights).sum()


In [ ]:
def flat_grad(policy, states, actions, weights):
    policy.zero_grad()
    policy_loss(policy, states, actions, weights).backward()
    return torch.cat([p.grad.flatten() for p in policy.parameters()]).clone()

def gradient_variance(env, policy, value, gamma, n, seed):
    """Total variance of single-episode gradient estimates, without and with
    the value baseline, for the current, frozen policy."""
    g_plain, g_base = [], []
    for i in range(n):
        states, actions, rewards = rollout(env, policy, seed=seed + 50_000 + i)
        G = rewards_to_go(rewards, gamma)
        with torch.no_grad():
            b = value(states).squeeze(1)
        g_plain.append(flat_grad(policy, states, actions, G))
        g_base.append(flat_grad(policy, states, actions, G - b))
    policy.zero_grad()
    tv = lambda g: torch.stack(g).var(dim=0).sum().item()
    return tv(g_plain), tv(g_base)


**Expected output**

Running `train_reinforce` for 1,000 episodes with seeds 0 to 4 (`variance_every=100`), with and without the baseline, and printing summary statistics gives:

```text
no baseline  mean return eps 1-200:   98.0  eps 201-500:  205.6  last 100:  365.5 (per seed [321, 494, 500, 109, 404]); first episode with 20-ep average >= 475: [608, 630, 480, 511, 988]
baseline     mean return eps 1-200:  134.7  eps 201-500:  345.6  last 100:  427.2 (per seed [500, 483, 484, 172, 496]); first episode with 20-ep average >= 475: [361, 248, 359, 277, 250]
```
